In [ ]:
!pip install duckdb -q

In [ ]:
import duckdb
import pandas as pd

df = duckdb.read_csv("/content/train.csv")

In [ ]:
dff = pd.read_csv("train.csv")

In [ ]:
dff.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9800 entries, 0 to 9799
Data columns (total 18 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Row ID         9800 non-null   int64  
 1   Order ID       9800 non-null   object 
 2   Order Date     9800 non-null   object 
 3   Ship Date      9800 non-null   object 
 4   Ship Mode      9800 non-null   object 
 5   Customer ID    9800 non-null   object 
 6   Customer Name  9800 non-null   object 
 7   Segment        9800 non-null   object 
 8   Country        9800 non-null   object 
 9   City           9800 non-null   object 
 10  State          9800 non-null   object 
 11  Postal Code    9789 non-null   float64
 12  Region         9800 non-null   object 
 13  Product ID     9800 non-null   object 
 14  Category       9800 non-null   object 
 15  Sub-Category   9800 non-null   object 
 16  Product Name   9800 non-null   object 
 17  Sales          9800 non-null   float64
dtypes: float

In [ ]:
dff.head()

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,State,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales
0,1,CA-2017-152156,08/11/2017,11/11/2017,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420.0,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600
1,2,CA-2017-152156,08/11/2017,11/11/2017,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420.0,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400
2,3,CA-2017-138688,12/06/2017,16/06/2017,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,California,90036.0,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200
3,4,US-2016-108966,11/10/2016,18/10/2016,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311.0,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775
4,5,US-2016-108966,11/10/2016,18/10/2016,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311.0,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680


In [ ]:
dff.nunique()

,0
Row ID,9800
Order ID,4922
Order Date,1230
Ship Date,1326
Ship Mode,4
Customer ID,793
Customer Name,793
Segment,3
Country,1
City,529


In [ ]:
dff.describe()

,Row ID,Postal Code,Sales
count,9800.000000,9789.000000,9800.000000
mean,4900.500000,55273.322403,230.769059
std,2829.160653,32041.223413,626.651875
min,1.000000,1040.000000,0.444000
25%,2450.750000,23223.000000,17.248000
50%,4900.500000,58103.000000,54.490000
75%,7350.250000,90008.000000,210.605000
max,9800.000000,99301.000000,22638.480000


In [ ]:
duckdb.sql("""
    SELECT
        category,
        SUM(sales) as total_sales
    FROM df
    group by category
    order by total_sales desc
""")

┌─────────────────┬───────────────────┐
│    Category     │    total_sales    │
│     varchar     │      double       │
├─────────────────┼───────────────────┤
│ Technology      │ 827455.8729999965 │
│ Furniture       │ 728658.5756999997 │
│ Office Supplies │ 705422.3340000033 │
└─────────────────┴───────────────────┘

**SQL QUESTIONING**

HOW MUCH

In [ ]:
# How many total rows are in the dataset?

duckdb.sql("""
    SELECT
        COUNT(*) as total_rows
    FROM df
""")

┌────────────┐
│ total_rows │
│   int64    │
├────────────┤
│       9800 │
└────────────┘

In [ ]:
# Total Sales?

duckdb.sql("""
    SELECT
        SUM(sales) as total_sales
    FROM df
""")

┌───────────────────┐
│    total_sales    │
│      double       │
├───────────────────┤
│ 2261536.782699953 │
└───────────────────┘

In [ ]:
# How many orders did we receive?

duckdb.sql("""
    SELECT
        count(distinct "Order ID") as total_orders
    FROM df
""")

┌──────────────┐
│ total_orders │
│    int64     │
├──────────────┤
│         4922 │
└──────────────┘

In [ ]:
# How many unique customers do we have?

duckdb.sql("""
    SELECT
        count(distinct "Customer ID") as total_customers
    FROM df
""")

┌─────────────────┐
│ total_customers │
│      int64      │
├─────────────────┤
│             793 │
└─────────────────┘

In [ ]:
# Average Sales per Order?

duckdb.sql("""
    select
        sum(sales) / count(distinct "Order ID") as avg_sales_per_order
    from df
""")

┌─────────────────────┐
│ avg_sales_per_order │
│       double        │
├─────────────────────┤
│  459.47516917918585 │
└─────────────────────┘

WHEN

In [ ]:
# Total sales for each month, ordered chronologically.

duckdb.sql("""
    select
        date_trunc('month', "Order Date") as month,
        sum(sales) as total_sales
    from df
    group by month
    order by month ASC
""")

┌────────────┬────────────────────┐
│   month    │    total_sales     │
│    date    │       double       │
├────────────┼────────────────────┤
│ 2015-01-01 │ 14205.706999999997 │
│ 2015-02-01 │           4519.892 │
│ 2015-03-01 │  55205.79700000003 │
│ 2015-04-01 │ 27906.854999999992 │
│ 2015-05-01 │          23644.303 │
│ 2015-06-01 │  34322.93560000002 │
│ 2015-07-01 │          33781.543 │
│ 2015-08-01 │ 27117.536499999995 │
│ 2015-09-01 │  81623.52679999998 │
│ 2015-10-01 │  31453.39299999999 │
│     ·      │          ·         │
│     ·      │          ·         │
│     ·      │          ·         │
│ 2018-03-01 │ 58863.412799999984 │
│ 2018-04-01 │  35541.91010000001 │
│ 2018-05-01 │  43825.98219999999 │
│ 2018-06-01 │         48190.7277 │
│ 2018-07-01 │  44825.10400000001 │
│ 2018-08-01 │  62837.84799999998 │
│ 2018-09-01 │  86152.88800000004 │
│ 2018-10-01 │  77448.13119999997 │
│ 2018-11-01 │ 117938.15500000001 │
│ 2018-12-01 │         83030.3888 │
├────────────┴──────────────

In [ ]:
# Total sales for each year, ordered chronologically.

duckdb.sql("""
    select
        date_trunc('year', "Order Date") as year,
        sum(sales) as total_sales
    from df
    group by year
    order by year ASC
""")

┌────────────┬───────────────────┐
│    year    │    total_sales    │
│    date    │      double       │
├────────────┼───────────────────┤
│ 2015-01-01 │  479856.208100001 │
│ 2016-01-01 │ 459436.0054000001 │
│ 2017-01-01 │  600192.550000001 │
│ 2018-01-01 │ 722052.0192000001 │
└────────────┴───────────────────┘

In [ ]:
# How much sales does each category generate in each year?

duckdb.sql("""
    select
        sum(sales) as total_sales,
        category,
        date_trunc('year', "Order Date") as year
    from df
    group by category, year
    order by year ASC
""")

┌────────────────────┬─────────────────┬────────────┐
│    total_sales     │    Category     │    year    │
│       double       │     varchar     │    date    │
├────────────────────┼─────────────────┼────────────┤
│ 156477.88110000003 │ Furniture       │ 2015-01-01 │
│ 149512.82000000007 │ Office Supplies │ 2015-01-01 │
│ 173865.50700000007 │ Technology      │ 2015-01-01 │
│  162257.7309999999 │ Technology      │ 2016-01-01 │
│ 164053.86740000008 │ Furniture       │ 2016-01-01 │
│ 133124.40699999977 │ Office Supplies │ 2016-01-01 │
│ 221961.94400000013 │ Technology      │ 2017-01-01 │
│ 195813.04000000018 │ Furniture       │ 2017-01-01 │
│ 182417.56599999993 │ Office Supplies │ 2017-01-01 │
│ 240367.54100000023 │ Office Supplies │ 2018-01-01 │
│ 212313.78720000002 │ Furniture       │ 2018-01-01 │
│ 269370.69100000017 │ Technology      │ 2018-01-01 │
├────────────────────┴─────────────────┴────────────┤
│ 12 rows                                 3 columns │
└───────────────────────────

In [ ]:
# How much sales does each region generate?

duckdb.sql("""
    select
        sum(sales) as total_sales,
        region
    from df
    group by region
    order by total_sales asc
""")

┌───────────────────┬─────────┐
│    total_sales    │ Region  │
│      double       │ varchar │
├───────────────────┼─────────┤
│ 389151.4590000004 │ South   │
│ 492646.9132000005 │ Central │
│ 669518.7259999984 │ East    │
│ 710219.6845000003 │ West    │
└───────────────────┴─────────┘

In [ ]:
# How much sales does each region generate in each category?

duckdb.sql("""
    select
        sum(sales) as total_sales,
        region,
        category
    from df
    group by region, category
    order by total_sales asc
""")

┌────────────────────┬─────────┬─────────────────┐
│    total_sales     │ Region  │    Category     │
│       double       │ varchar │     varchar     │
├────────────────────┼─────────┼─────────────────┤
│  116531.4800000001 │ South   │ Furniture       │
│ 124424.77099999994 │ South   │ Office Supplies │
│ 148195.20799999993 │ South   │ Technology      │
│ 160317.46220000004 │ Central │ Furniture       │
│ 163590.24300000028 │ Central │ Office Supplies │
│ 168739.20799999993 │ Central │ Technology      │
│ 199940.81099999984 │ East    │ Office Supplies │
│ 206461.38800000006 │ East    │ Furniture       │
│ 217466.50900000005 │ West    │ Office Supplies │
│  245348.2455000003 │ West    │ Furniture       │
│          247404.93 │ West    │ Technology      │
│ 263116.52700000035 │ East    │ Technology      │
├────────────────────┴─────────┴─────────────────┤
│ 12 rows                              3 columns │
└────────────────────────────────────────────────┘

In [ ]:
# Analyze orders by region

duckdb.sql("""
    select
        count(distinct "Order ID") as total_orders,
        region
    from df
    group by region
    order by total_orders asc
""")

┌──────────────┬─────────┐
│ total_orders │ Region  │
│    int64     │ varchar │
├──────────────┼─────────┤
│          810 │ South   │
│         1156 │ Central │
│         1369 │ East    │
│         1587 │ West    │
└──────────────┴─────────┘

In [ ]:
# Calculate average sales per order by region.

duckdb.sql("""
    select
        sum(sales) / count(distinct "Order ID") as avg_sales_per_order,
        region
    from df
    group by region
    order by avg_sales_per_order asc
""")

┌─────────────────────┬─────────┐
│ avg_sales_per_order │ Region  │
│       double        │ varchar │
├─────────────────────┼─────────┤
│  426.16514982699005 │ Central │
│   447.5234306868307 │ West    │
│  480.43390000000045 │ South   │
│   489.0567757487205 │ East    │
└─────────────────────┴─────────┘

In [ ]:
# Customer base by region

duckdb.sql("""
    select
        count(distinct "Customer ID") as total_customers,
        region
    from df
    group by region
    order by total_customers asc
""")

┌─────────────────┬─────────┐
│ total_customers │ Region  │
│      int64      │ varchar │
├─────────────────┼─────────┤
│             509 │ South   │
│             626 │ Central │
│             669 │ East    │
│             681 │ West    │
└─────────────────┴─────────┘

In [ ]:
duckdb.sql("""
    select
        count(distinct "Customer ID") as total_customers,
        count(distinct "Order ID") as total_orders,
        sum(sales) as total_sales,
        sum(sales) / count(distinct "Order ID") as avg_sales_per_order,
        segment,
        region
    from df
    group by region, segment
    order by total_sales asc
""")

┌─────────────────┬──────────────┬────────────────────┬─────────────────────┬─────────────┬─────────┐
│ total_customers │ total_orders │    total_sales     │ avg_sales_per_order │   Segment   │ Region  │
│      int64      │    int64     │       double       │       double        │   varchar   │ varchar │
├─────────────────┼──────────────┼────────────────────┼─────────────────────┼─────────────┼─────────┤
│              82 │          128 │  73902.37150000005 │   577.3622773437504 │ Home Office │ South   │
│             120 │          221 │  90404.89439999995 │   409.0719203619907 │ Home Office │ Central │
│             152 │          244 │ 120546.87450000003 │    494.044567622951 │ Corporate   │ South   │
│             124 │          247 │         125714.696 │   508.9663805668016 │ Home Office │ East    │
│             123 │          298 │ 134960.21499999994 │  452.88662751677833 │ Home Office │ West    │
│             180 │          345 │        152031.4968 │  440.67100521739127 │ Corp

In [ ]:
# Total sales by Sub-Category, ordered from highest to lowest.

duckdb.sql("""
    select
        sum(sales) as total_sales,
        "Sub-Category"
    from df
    group by "Sub-Category"
    order by total_sales desc
""")

┌────────────────────┬──────────────┐
│    total_sales     │ Sub-Category │
│       double       │   varchar    │
├────────────────────┼──────────────┤
│ 327782.44800000027 │ Phones       │
│  322822.7310000008 │ Chairs       │
│  219343.3920000001 │ Storage      │
│  202810.6280000001 │ Tables       │
│ 200028.78500000003 │ Binders      │
│ 189238.63099999996 │ Machines     │
│ 164186.70000000016 │ Accessories  │
│ 146248.09399999995 │ Copiers      │
│ 113813.19869999998 │ Bookcases    │
│ 104618.40299999998 │ Appliances   │
│  89212.01800000003 │ Furnishings  │
│  76828.30400000002 │ Paper        │
│  46420.30800000001 │ Supplies     │
│ 26705.409999999956 │ Art          │
│ 16128.045999999997 │ Envelopes    │
│          12347.726 │ Labels       │
│ 3001.9599999999996 │ Fasteners    │
├────────────────────┴──────────────┤
│ 17 rows                 2 columns │
└───────────────────────────────────┘

In [ ]:
# Top 10 products by sales

duckdb.sql("""
    select
        sum(sales) as total_sales,
        "Product Name"
    from df
    group by "Product Name"
    order by total_sales desc
    limit 10
""")

┌────────────────────┬─────────────────────────────────────────────────────────────────────────────┐
│    total_sales     │                                Product Name                                 │
│       double       │                                   varchar                                   │
├────────────────────┼─────────────────────────────────────────────────────────────────────────────┤
│          61599.824 │ Canon imageCLASS 2200 Advanced Copier                                       │
│          27453.384 │ Fellowes PB500 Electric Punch Plastic Comb Binding Machine with Manual Bind │
│           22638.48 │ Cisco TelePresence System EX90 Videoconferencing Unit                       │
│          21870.576 │ HON 5400 Series Task Chairs for Big and Tall                                │
│ 19823.479000000003 │ GBC DocuBind TL300 Electric Binding System                                  │
│            19024.5 │ GBC Ibimaster 500 Manual ProClick Binding System                    

In [ ]:
# What percentage of total sales comes from the top 10 products?

duckdb.sql("""
    with top_products as(
            select
            sum(sales) as total_sales,
            "Product Name"
        from df
        group by "Product Name"
        order by total_sales desc
        limit 10
    )
    select
        sum(total_sales) / (select sum(sales) from df) * 100 as top_10_sales_percentage
    from top_products
""")

┌─────────────────────────┐
│ top_10_sales_percentage │
│         double          │
├─────────────────────────┤
│      10.816547662247542 │
└─────────────────────────┘

In [ ]:
# How much of our total sales comes from our top 10 customers?

duckdb.sql("""
    with top_customers as(
            select
                sum(sales) as customer_sales,
                "Customer ID"
            from df
            group by "Customer ID"
            order by customer_sales desc
            limit 10
    )
    select
        sum(customer_sales) / (select sum(sales) from df) * 100 as top_10_customers_percentage
    from top_customers
""")

┌─────────────────────────────┐
│ top_10_customers_percentage │
│           double            │
├─────────────────────────────┤
│           6.801179188267341 │
└─────────────────────────────┘